# Wind & Solar Capacity Predictor

Target => val_geracaoverificada

In [2]:
%pip install duckdb numpy pandas scikit-learn pyarrow xgboost matplotlib torch tabpfn


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import gc
import duckdb
import pyarrow
import numpy as np
import pandas as pd
from os import listdir
from pathlib import Path
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.neighbors import BallTree
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
import torch
from tabpfn import TabPFNRegressor

Agregando os dados extraidos da [ONS](https://dados.ons.org.br/dataset/fator-capacidade-2):

In [4]:
df = pd.concat(
    [pd.read_parquet(f"./database/{file}") for file in listdir("./database") if file.endswith('.parquet')],
    ignore_index=True
)

Ajustando a tipagem das colunas do DF de acordo com a documentação:

In [5]:
float_columns = [
    "val_latitudesecoletora",
    "val_longitudesecoletora",
    "val_latitudepontoconexao",
    "val_longitudepontoconexao",
    "val_geracaoverificada",
    "val_capacidadeinstalada",
    "val_fatorcapacidade"
]

df[float_columns] = df[float_columns].apply(pd.to_numeric, errors="coerce")

df["din_instante"] = pd.to_datetime(df["din_instante"])

### Setup MatPlotLib e DuckDB

In [ ]:
con = duckdb.connect()
con.register("ons", df) 

pd.set_option('display.max_rows', None)

In [ ]:
SUPERFICIE = "#fcfcfb"
TINTA      = "#0b0b0b"
TINTA_2    = "#52514e"
MUTED      = "#898781"
GRADE      = "#e1e0d9"
EIXO       = "#c3c2b7"
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
         "#e87ba4", "#008300", "#6250d6", "#e34948"]
 
plt.rcParams.update({
    "figure.facecolor": SUPERFICIE, "axes.facecolor": SUPERFICIE, "savefig.facecolor": SUPERFICIE,
    "axes.edgecolor": EIXO, "axes.labelcolor": TINTA_2, "text.color": TINTA,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRADE, "grid.linewidth": 0.6,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "lines.linewidth": 2, "legend.frameon": False, "figure.dpi": 110,
})
 
# Escala sequencial (uma cor só, do claro ao escuro) para os mapas de calor
AZUIS = LinearSegmentedColormap.from_list(
    "azuis", ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"])
AZUIS.set_bad(SUPERFICIE)   # células com zero ficam da cor do fundo
 
# Cada fonte tem sempre a mesma cor em todos os gráficos
CORES_FONTE = {"Eólica": SLOTS[0], "Solar": SLOTS[1]}
def cor_fonte(fonte):
    if fonte not in CORES_FONTE:
        CORES_FONTE[fonte] = SLOTS[len(CORES_FONTE)]
    return CORES_FONTE[fonte]
 
MESES = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]

### Consulta SQL 1) Evolução anual da capacidade e da geração por fonte

Mostra como o parque eólico e o solar cresceram ao longo dos anos, em capacidade instalada e em energia gerada, e quanto cada fonte representa da geração total de cada ano.

**Retorna:** uma linha por ano e fonte, com as colunas:
- `ano`, `nom_tipousina`: ano e fonte (Eólica / Fotovoltaica)
- `n_usinas`: quantidade de usinas daquele tipo ativas no ano
- `capacidade_mw`: capacidade instalada total no fim do ano (MW)
- `geracao_twh`: energia gerada no ano (TWh)
- `cresc_cap_anual`, `cresc_ger_anul`: crescimento da capacidade e da geração em relação ao ano anterior (0,25 = +25%); vazio no primeiro ano de cada fonte
- `prop_anual`: participação em porcentagem da fonte na geração do ano (0 a 1)

In [ ]:
q1 = con.sql("""
WITH usina_ano AS (
    SELECT year(din_instante) AS ano, nom_tipousina, id_ons,
           arg_max(val_capacidadeinstalada, din_instante) AS cap_fim_ano_mw,
           SUM(val_geracaoverificada)                     AS geracao_mwh
    FROM ons
    GROUP BY ALL
),
anual AS (
    SELECT ano, nom_tipousina,
           COUNT(*)               AS n_usinas,
           SUM(cap_fim_ano_mw)    AS capacidade_mw,
           SUM(geracao_mwh) / 1e6 AS geracao_twh
    FROM usina_ano
    GROUP BY ALL
)
SELECT *,
       capacidade_mw / LAG(capacidade_mw) OVER w - 1        AS cresc_cap_anual,
       geracao_twh   / LAG(geracao_twh)   OVER w - 1        AS cresc_ger_anual,
       geracao_twh / SUM(geracao_twh) OVER (PARTITION BY ano) AS prop_anual
FROM anual
WINDOW w AS (PARTITION BY nom_tipousina ORDER BY ano)
ORDER BY nom_tipousina, ano
""").df()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for fonte, g in q1.groupby("nom_tipousina"):
    g = g.sort_values("ano")
    ax.plot(g["ano"], g["geracao_twh"], color=cor_fonte(fonte),
            marker="o", markersize=6, label=fonte)
    ax.annotate(f"{fonte}  {g['geracao_twh'].iloc[-1]:.1f} TWh",
                (g["ano"].iloc[-1], g["geracao_twh"].iloc[-1]),
                xytext=(8, 0), textcoords="offset points", va="center", color=TINTA_2)
 
ax.set_title("Geração anual por fonte")
ax.set_ylabel("TWh")
ax.set_ylim(bottom=0)
ax.set_xlim(q1["ano"].min() - 0.5, q1["ano"].max() + 2.5)   # espaço para os rótulos
ax.set_xticks(range(q1["ano"].min(), q1["ano"].max() + 1, 2))
ax.grid(axis="x", visible=False)
ax.legend(loc="upper left")
fig.text(0.01, -0.02, "O primeiro e o último ano da base podem estar incompletos.",
         color=MUTED, fontsize=9)
plt.show()

In [ ]:
share = (q1.pivot(index="ano", columns="nom_tipousina", values="prop_anual")
           .fillna(0).sort_index() * 100)
 
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.stackplot(share.index, share.T.values, labels=share.columns,
             colors=[cor_fonte(c) for c in share.columns],
             edgecolor=SUPERFICIE, linewidth=2)
 
# Rótulo direto no último ano, dentro de cada faixa
base = 0
for fonte, v in share.iloc[-1].items():
    if v >= 5:
        ax.text(share.index[-1] - 0.2, base + v / 2, f"{fonte}  {v:.0f}%",
                ha="right", va="center", color="white", fontweight="bold")
    base += v
 
ax.set_title("Participação de cada fonte na geração anual")
ax.set_ylim(0, 100)
ax.set_xlim(share.index.min(), share.index.max())
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.grid(axis="x", visible=False)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.show()

### Consulta SQL 2) Sazonalidade do fator de capacidade

Mostra o aproveitamento médio das usinas (geração ÷ capacidade) mês a mês, por fonte e região, e suaviza a série com médias móveis para evidenciar o ciclo anual e a tendência de longo prazo.

**Retorna:** uma linha por mês, fonte e subsistema, com as colunas:
- `mes`, `nom_tipousina`, `nom_subsistema`: mês, fonte e região do sistema elétrico
- `fc`: fator de capacidade do mês (0 a 1; 0,45 = a fonte gerou 45% do máximo possível)
- `fc_mm3`: média móvel de 3 meses, que mostra o ciclo sazonal
- `fc_mm12`: média móvel de 12 meses, que mostra a tendência sem a sazonalidade

In [ ]:
q2 = con.sql("""
WITH mensal AS (
    SELECT date_trunc('month', din_instante) AS mes,
           nom_tipousina, nom_subsistema,
           SUM(val_geracaoverificada) / NULLIF(SUM(val_capacidadeinstalada), 0) AS fc
    FROM ons
    WHERE val_capacidadeinstalada > 0
    GROUP BY ALL
)
SELECT *,
       AVG(fc) OVER (PARTITION BY nom_tipousina, nom_subsistema ORDER BY mes
                     ROWS BETWEEN 2  PRECEDING AND CURRENT ROW) AS fc_mm3,
       AVG(fc) OVER (PARTITION BY nom_tipousina, nom_subsistema ORDER BY mes
                     ROWS BETWEEN 11 PRECEDING AND CURRENT ROW) AS fc_mm12
FROM mensal
ORDER BY nom_tipousina, nom_subsistema, mes
""").df()

In [ ]:
fontes = sorted(q2["nom_tipousina"].unique())
subs = sorted(q2["nom_subsistema"].dropna().unique())
cores_sub = dict(zip(subs, SLOTS))
 
fig, axes = plt.subplots(len(fontes), 1, figsize=(11, 3.8 * len(fontes)),
                         sharex=True, squeeze=False)
for ax, fonte in zip(axes[:, 0], fontes):
    d = q2[q2["nom_tipousina"] == fonte]
    for sub, g in d.groupby("nom_subsistema"):
        g = g.sort_values("mes")
        ax.plot(g["mes"], g["fc_mm12"], color=cores_sub[sub], linewidth=2.2, label=sub)
    ax.set_title(f"{fonte}: fator de capacidade mensal")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.set_ylim(bottom=0)
    ax.grid(axis="x", visible=False)
    ax.legend(title="Subsistema", loc="upper left", bbox_to_anchor=(1.01, 1))
 
fig.text(0.01, -0.01,"Média móvel de 12 meses (tendência).", color=MUTED, fontsize=9)
plt.tight_layout()
plt.show()

### Consulta SQL 3) Mês e hora de pico da geração por usina e ano

Mostra, para cada usina, em que mês e em que hora do dia ela teve sua maior geração média em cada ano. Permite ver se o momento de melhor aproveitamento é estável ao longo dos anos e como ele difere entre eólicas e solares.

**Retorna:** uma linha por usina e ano, com as colunas:
- `id_ons`, `nom_usina_conjunto`, `nom_tipousina`: usina e fonte
- `ano`: ano de referência
- `mes_pico`: mês (1 a 12) em que ocorreu o pico de geração
- `hora_pico`: hora do dia (0 a 23) do pico dentro desse mês
- `fc_pico`: fator de capacidade médio nessa combinação de mês e hora (0 a 1)

In [ ]:
q3 = con.sql("""
WITH perfil AS (
    SELECT id_ons, nom_usina_conjunto, nom_tipousina,
           year(din_instante)  AS ano,
           month(din_instante) AS mes,
           hour(din_instante)  AS hora,
           SUM(val_geracaoverificada) / SUM(val_capacidadeinstalada) AS fc
    FROM ons
    WHERE val_capacidadeinstalada > 0
    GROUP BY ALL
)
SELECT id_ons, nom_usina_conjunto, nom_tipousina, ano,
       mes  AS mes_pico,
       hora AS hora_pico,
       ROUND(fc, 3) AS fc_pico
FROM perfil
QUALIFY ROW_NUMBER() OVER (PARTITION BY id_ons, ano ORDER BY fc DESC) = 1
ORDER BY id_ons, ano
""").df()

In [ ]:
fontes = sorted(q3["nom_tipousina"].unique())
 
fig, axes = plt.subplots(1, len(fontes), figsize=(6.5 * len(fontes), 6.5),
                         squeeze=False)   # cada painel com sua própria coluna de horas
for ax, fonte in zip(axes[0], fontes):
    d = q3[q3["nom_tipousina"] == fonte]
    tab = (pd.crosstab(d["hora_pico"], d["mes_pico"])
             .reindex(index=range(24), columns=range(1, 13), fill_value=0))
    dados = np.ma.masked_equal(tab.to_numpy(), 0)
 
    m = ax.pcolormesh(np.arange(13), np.arange(25), dados, cmap=AZUIS,
                      edgecolors=SUPERFICIE, linewidth=1)
    ax.set_xticks(np.arange(12) + 0.5, MESES)
    ax.set_yticks(np.arange(0, 24, 2) + 0.5, [f"{h}h" for h in range(0, 24, 2)])
    ax.invert_yaxis()
    ax.grid(False)
    ax.spines[:].set_visible(False)
    ax.set_title(fonte)
    cb = fig.colorbar(m, ax=ax, shrink=0.8)
    cb.set_label("nº de usinas x anos com pico em cada posição")
    cb.outline.set_visible(False)
 
fig.suptitle("Em que mês e hora cada usina atinge seu pico de geração",
             x=0.01, ha="left", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Consulta SQL 4) Paradas prolongadas de usinas eólicas

Lista os períodos em que usinas eólicas ficaram 24 horas ou mais seguidas sem gerar, mostrando as 5 maiores paradas de cada ano. São eventos que o clima não explica (manutenção, restrição, falha de medição) e que podem ser tratados antes do treino do modelo.

**Retorna:** até 5 linhas por ano, com as colunas:
- `ano`: ano em que a parada começou
- `id_ons`: usina
- `inicio`, `fim`: primeira e última hora zerada
- `horas_zerada`: quantidade de horas registradas com geração zero
- `horas_calendario`: horas corridas entre início e fim
- `tem_buraco_nos_dados`: `True` quando há horas sem registro no meio da parada (o período inclui falta de dados, não só geração zero)

In [ ]:
q4 = con.sql("""
WITH flag AS (
    SELECT id_ons, din_instante,
           (val_geracaoverificada <= 0) AS zerada
    FROM ons
    WHERE nom_tipousina = 'Eólica'
),
ilhas AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY id_ons ORDER BY din_instante)
         - ROW_NUMBER() OVER (PARTITION BY id_ons, zerada ORDER BY din_instante) AS grp
    FROM flag
),
paradas AS (
    SELECT id_ons,
           MIN(din_instante) AS inicio,
           MAX(din_instante) AS fim,
           COUNT(*)          AS horas_zerada,
           date_diff('hour', MIN(din_instante), MAX(din_instante)) + 1 AS horas_calendario
    FROM ilhas
    WHERE zerada
    GROUP BY id_ons, grp
    HAVING COUNT(*) >= 24
)
SELECT year(inicio) AS ano, *,
       horas_calendario > horas_zerada AS tem_buraco_nos_dados
FROM paradas
QUALIFY ROW_NUMBER() OVER (PARTITION BY year(inicio) ORDER BY horas_zerada DESC) <= 5
ORDER BY ano, horas_zerada DESC
""").df()

In [ ]:
d = (q4.sort_values(["ano", "horas_zerada"], ascending=[True, False])
       .reset_index(drop=True))
d["dias"] = d["horas_zerada"] / 24
d["rotulo"] = d["id_ons"] + "  ·  " + pd.to_datetime(d["inicio"]).dt.strftime("%d/%m/%Y")
 
fig, ax = plt.subplots(figsize=(10, max(4, 0.3 * len(d))))
y = np.arange(len(d))
barras = ax.barh(y, d["dias"], height=0.7,
                 color=np.where(d["tem_buraco_nos_dados"], MUTED, SLOTS[0]))
for b, buraco in zip(barras, d["tem_buraco_nos_dados"]):
    if buraco:                       # textura, para não depender só da cor
        b.set_hatch("//")
        b.set_edgecolor(SUPERFICIE)
 
ax.set_yticks(y, d["rotulo"], fontsize=8)
ax.set_ylim(len(d) - 0.5, -0.5)      # de cima para baixo, sem sobra nas pontas
ax.grid(axis="y", visible=False)
ax.set_xlabel("Duração da parada (dias)")
 
# Separador e rótulo de cada ano
for ano, g in d.groupby("ano"):
    ax.axhline(g.index.max() + 0.5, color=EIXO, linewidth=0.8)
    ax.text(1.01, g.index.min() + (len(g) - 1) / 2, str(ano),
            transform=ax.get_yaxis_transform(), va="center",
            color=TINTA_2, fontweight="bold")
 
ax.legend(handles=[Patch(color=SLOTS[0], label="Parada contínua"),
                   Patch(facecolor=MUTED, edgecolor=SUPERFICIE, hatch="//",
                         label="Atravessa período sem dados")],
          loc="lower left", bbox_to_anchor=(0, 1), borderaxespad=0.2, ncol=2)
ax.set_title("Maiores paradas de usinas eólicas por ano", pad=34)
plt.show()

### Consulta SQL 5) Variabilidade horária e baseline de persistência

Mede o quanto a geração oscila de uma hora para outra e quanto erraria uma previsão ingênua que repete a geração do dia anterior na mesma hora. Esse erro serve de referência mínima que o modelo deve superar.

**Retorna:** uma linha por fonte, ano e trimestre, com as colunas:
- `nom_tipousina`, `ano`, `trimestre`: fonte e período
- `rampa_media_pct_cap`: variação média entre horas seguidas, em % da capacidade
- `rampa_p95_pct_cap`: variação nas 5% maiores oscilações, em % da capacidade
- `wape_persistencia_24h`: erro percentual da previsão "amanhã = hoje" (%)
- `delta_wape_vs_ano_ant`: diferença desse erro em relação ao mesmo trimestre do ano anterior, em pontos percentuais; vazio no primeiro ano

In [ ]:
q5 = con.sql("""
WITH serie AS (
    SELECT nom_tipousina, din_instante,
           val_geracaoverificada   AS g,
           val_capacidadeinstalada AS cap,
           LAG(val_geracaoverificada, 1)  OVER w AS g_1h,
           LAG(val_geracaoverificada, 24) OVER w AS g_24h
    FROM ons
    WINDOW w AS (PARTITION BY id_ons ORDER BY din_instante)
),
trimestral AS (
    SELECT nom_tipousina,
           year(din_instante)    AS ano,
           quarter(din_instante) AS trimestre,
           AVG(ABS(g - g_1h) / NULLIF(cap, 0)) * 100                 AS rampa_media_pct_cap,
           quantile_cont(ABS(g - g_1h) / NULLIF(cap, 0), 0.95) * 100 AS rampa_p95_pct_cap,
           SUM(ABS(g - g_24h)) / NULLIF(SUM(g), 0) * 100             AS wape_persistencia_24h
    FROM serie
    GROUP BY ALL
)
SELECT *,
       wape_persistencia_24h
         - LAG(wape_persistencia_24h) OVER (PARTITION BY nom_tipousina, trimestre ORDER BY ano)
         AS delta_wape_vs_ano_ant
FROM trimestral
ORDER BY nom_tipousina, ano, trimestre
""").df()

In [ ]:
d = q5.copy()
d["periodo"] = pd.to_datetime(pd.DataFrame({
    "year": d["ano"], "month": (d["trimestre"] - 1) * 3 + 1, "day": 1}))
 
fig, ax = plt.subplots(figsize=(11, 4.5))
for fonte, g in d.groupby("nom_tipousina"):
    g = g.sort_values("periodo")
    ax.plot(g["periodo"], g["wape_persistencia_24h"], color=cor_fonte(fonte),
            marker="o", markersize=5, label=fonte)
 
ax.set_title("Erro da previsão por persistência (amanhã = hoje), por trimestre")
ax.set_ylabel("WAPE (%)")
ax.set_ylim(bottom=0)
ax.grid(axis="x", visible=False)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
 
plt.show()

In [ ]:
df = df.drop(['nom_subsistema', 'cod_pontoconexao', 'ceg', 'nom_estado', 'val_fatorcapacidade', 'val_geracaoprogramada'],axis=1)

ano_alvo = 2012

Lê os dados metereológicos extraídos do [INMET](https://portal.inmet.gov.br/dadoshistoricos) a partir de 2009 até 2026:

In [ ]:
def _num(valor):
    """Converte '1159,54' -> 1159.54; qualquer valor inválido vira NaN."""
    return pd.to_numeric(str(valor).replace(",", "."), errors="coerce")

def ler_inmet(caminho):
    # 1) Metadados (8 primeiras linhas)
    meta = {}
    with open(caminho, encoding="latin-1") as f:
        for _ in range(8):
            chave, _, valor = f.readline().strip().partition(";")
            meta[chave.rstrip(":")] = valor.strip()

    # 2) Dados
    df = pd.read_csv(
        caminho,
        sep=";",
        skiprows=8,
        decimal=",",
        encoding="latin-1",
        na_values=["-9999"],
        index_col=False,
    )
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    df["REGIAO"] = meta.get("REGIÃO")
    df["UF"] = meta.get("UF")
    df["ESTACAO"] = meta.get("ESTAÇÃO")
    df["CODIGO_WMO"] = meta.get("CODIGO (WMO)")
    df["LATITUDE"] = _num(meta.get("LATITUDE"))
    df["LONGITUDE"] = _num(meta.get("LONGITUDE"))
    df["ALTITUDE"] = _num(meta.get("ALTITUDE"))
    df["DATA_FUNDACAO"] = pd.to_datetime(
        meta.get("DATA DE FUNDAÇÃO (YYYY-MM-DD)"), errors="coerce"
    )

    return df

pasta = Path(f"./database/INMET/{ano_alvo}")

df_weather = pd.concat(
    [ler_inmet(pasta / f) for f in listdir(pasta) if f.lower().endswith(".csv")],
    ignore_index=True,
)

In [ ]:
def padronizar_colunas(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza nomes de colunas que mudaram ao longo dos anos no INMET.

    `rename` ignora chaves ausentes, então não depende de cortes de ano.
    """
    return df.rename(columns={
        # formato até 2018
        "DATA (YYYY-MM-DD)": "DATA",
        "HORA (UTC)": "HORA",
        # formato a partir de 2019
        "Data": "DATA",
        "Hora UTC": "HORA",
        # radiação (KJ vs Kj)
        "RADIACAO GLOBAL (KJ/m²)": "RADIACAO GLOBAL (Kj/m²)",
    })


FLOAT_COLUMNS = [
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",
    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",
    "PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)",
    "PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)",
    "RADIACAO GLOBAL (Kj/m²)",
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",
    "TEMPERATURA DO PONTO DE ORVALHO (°C)",
    "TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)",
    "UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)",
    "UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)",
    "UMIDADE RELATIVA DO AR, HORARIA (%)",
    "VENTO, DIREÇÃO HORARIA (gr) (° (gr))",
    "VENTO, RAJADA MAXIMA (m/s)",
    "VENTO, VELOCIDADE HORARIA (m/s)",
    "LATITUDE",
    "LONGITUDE",
    "ALTITUDE",
]
CATEGORY_COLUMNS = ["REGIAO", "UF", "ESTACAO", "CODIGO_WMO"]

df_weather = padronizar_colunas(df_weather)

df_weather[FLOAT_COLUMNS] = df_weather[FLOAT_COLUMNS].apply(pd.to_numeric, errors="coerce")

hora = (
    df_weather["HORA"]
    .astype(str)
    .str.replace(" UTC", "", regex=False)
    .str.replace(":", "", regex=False)
    .str.zfill(4)
)
data = df_weather["DATA"].astype(str).str.replace("/", "-", regex=False)  # aceita 2019/01/01 e 2019-01-01

df_weather["DATETIME"] = pd.to_datetime(
    data + " " + hora,
    format="%Y-%m-%d %H%M",
    errors="coerce",
    utc=True,
)
df_weather["DATA_FUNDACAO"] = pd.to_datetime(df_weather["DATA_FUNDACAO"], errors="coerce")

df_weather = df_weather.drop(columns=["DATA", "HORA"])
df_weather[CATEGORY_COLUMNS] = df_weather[CATEGORY_COLUMNS].astype("category")

Join dos Datasets:

In [ ]:
# O INMET aqui é só de 20XX, então filtra o df para o mesmo ano
df_anual = df[df["din_instante"].dt.year == ano_alvo].copy()

# Garbage collector
del df
gc.collect()

In [ ]:
# ------------------------------------------------------------
# Prepara o weather
# ------------------------------------------------------------
weather = df_weather.copy()

weather["CODIGO_WMO"] = weather["CODIGO_WMO"].astype(str)
weather = weather.drop_duplicates(["CODIGO_WMO", "DATETIME"])

# Garbage collector
del df_weather
gc.collect()

In [ ]:
# ------------------------------------------------------------
# 1) Usina -> estação mais próxima
# ------------------------------------------------------------
plantas = (
    df_anual[["id_ons", "val_latitudesecoletora", "val_longitudesecoletora"]]
    .dropna()
    .drop_duplicates("id_ons")
    .reset_index(drop=True)
)

estacoes = (
    weather[["CODIGO_WMO", "LATITUDE", "LONGITUDE"]]
    .dropna()
    .drop_duplicates("CODIGO_WMO")
    .reset_index(drop=True)
)

tree = BallTree(
    np.radians(estacoes[["LATITUDE", "LONGITUDE"]].to_numpy()),
    metric="haversine",
)
dist, idx = tree.query(
    np.radians(plantas[["val_latitudesecoletora", "val_longitudesecoletora"]].to_numpy()),
    k=1,
)

plantas["CODIGO_WMO"] = estacoes["CODIGO_WMO"].to_numpy()[idx[:, 0]]
plantas["dist_estacao_km"] = dist[:, 0] * 6371  # raio da Terra em km

# ------------------------------------------------------------
# 2) Alinha o tempo (ONS = horário de Brasília, INMET = UTC)
# ------------------------------------------------------------
df_anual["DATETIME"] = (
    (df_anual["din_instante"] + pd.Timedelta(hours=3))
    .dt.tz_localize("UTC")
    .astype("datetime64[ns, UTC]")
)
weather["DATETIME"] = weather["DATETIME"].astype("datetime64[ns, UTC]")

# ------------------------------------------------------------
# 3) Join
# ------------------------------------------------------------
df_anual = df_anual.merge(
    plantas[["id_ons", "CODIGO_WMO", "dist_estacao_km"]],
    on="id_ons",
    how="left",
)

weather = weather.rename(columns={
    "LATITUDE": "LAT_ESTACAO",
    "LONGITUDE": "LON_ESTACAO",
    "ALTITUDE": "ALT_ESTACAO",
})

df_final = df_anual.merge(
    weather,
    on=["CODIGO_WMO", "DATETIME"],
    how="left",
    validate="many_to_one",
)

In [ ]:
del df_anual
del weather
gc.collect()

In [ ]:
df_final = df_final.drop(['val_latitudepontoconexao', 'val_longitudepontoconexao', 'LAT_ESTACAO', 'LON_ESTACAO', 'id_subsistema', 'id_estado', 'CODIGO_WMO'],axis=1)

In [ ]:
print(df_final.iloc[67])

# Modelagem:

In [ ]:
df_modelo = (
    df_final
    .dropna(subset=['val_geracaoverificada'])
    .sort_values(['din_instante', 'id_ons'], kind='stable')
    .reset_index(drop=True)
)

In [ ]:
del df_final
gc.collect()

In [ ]:
df_modelo['hora'] = df_modelo['din_instante'].dt.hour
df_modelo['dia_semana'] = df_modelo['din_instante'].dt.dayofweek
df_modelo['idade_estacao_anos'] = (
    (df_modelo['din_instante'].dt.tz_localize(None)
     - df_modelo['DATA_FUNDACAO'].dt.tz_localize(None)).dt.days / 365.25
)

y = df_modelo['val_geracaoverificada']

X = df_modelo.drop(columns=[
    'val_geracaoverificada',
    'din_instante', 'DATETIME', 'DATA_FUNDACAO',
    'nom_tipousina',
    'id_ons',
])

X_train, y_train = X, y

In [ ]:
N_FOLDS = 4
PCT_TESTE = 0.10 

datas_tr = pd.Series(pd.to_datetime(df_modelo['din_instante']).to_numpy())
gap_cv = pd.Timedelta(hours=24)

instantes = pd.to_datetime(np.sort(datas_tr.unique()))
n = len(instantes)
bordas = [k * n // N_FOLDS for k in range(N_FOLDS + 1)]

cv_splits = []
for k in range(N_FOLDS):
    bloco = instantes[bordas[k]:bordas[k + 1]]
    ini = bloco[0]
    fim = bloco[-1] + pd.Timedelta(seconds=1)
    ini_teste = bloco[int(len(bloco) * (1 - PCT_TESTE))]

    idx_tr = np.where((datas_tr >= ini) & (datas_tr < ini_teste - gap_cv))[0]
    idx_te = np.where((datas_tr >= ini_teste) & (datas_tr < fim))[0]
    assert len(idx_tr) > 0 and len(idx_te) > 0, f"Fold vazio no bloco iniciado em {ini}"
    cv_splits.append((idx_tr, idx_te))

print(f"{len(cv_splits)} folds:")
for i, (tr, te) in enumerate(cv_splits, 1):
    print(f"  fold {i}: treino {datas_tr[tr].min():%Y-%m-%d} -> {datas_tr[tr].max():%Y-%m-%d} ({len(tr):,}) | "
          f"teste {datas_tr[te].min():%Y-%m-%d} -> {datas_tr[te].max():%Y-%m-%d} ({len(te):,})")

## Baseline:

## Modelos Obrigatórios: Random Forest e SVR

### Random Forest:

In [ ]:
cat_cols = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
    ('num', 'passthrough', num_cols),
])

pipeline = Pipeline([
    ('prep', preprocess),
    ('rf', RandomForestRegressor(random_state=42, n_jobs=-1)),
])

param_grid = {
    'rf__n_estimators':      [200], # [100, 150, 200, 220]
    'rf__max_features':      ['sqrt'], # [sqrt, log2, 1.0]
    'rf__bootstrap':         [True],
    'rf__criterion':         ['squared_error'], # ['squared_error', 'absolute_error', 'poisson']
    'rf__max_depth':         [20], # [10, 15, 20, 22]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv_splits,
    scoring='neg_root_mean_squared_error',
    n_jobs=4,
    verbose=1
)

In [ ]:
grid_search.fit(X_train, y_train)
print(f"Melhores parâmetros: {grid_search.best_params_}")

In [ ]:
# ------------------------------------------------------------
# Resultado do CV
# ------------------------------------------------------------
print(f"Melhores parâmetros: {grid_search.best_params_}")
print(f"RMSE médio no CV (treino): {-grid_search.best_score_:.4f}")
print(f"Ano alvo: {ano_alvo}\n")

# ------------------------------------------------------------
# Predição fora da amostra: refaz cada fold com os melhores parâmetros
# e junta os testes dos 5 folds (não há hold-out final)
# ------------------------------------------------------------
yp, yt, cp, yb = [], [], [], []
for tr, te in cv_splits:
    m = clone(grid_search.best_estimator_)
    m.fit(X_train.iloc[tr], y_train.iloc[tr])
    yp.append(np.clip(m.predict(X_train.iloc[te]), 0, None))   # geração não é negativa
    yt.append(y_train.iloc[te].to_numpy())
    cp.append(X_train.iloc[te]['val_capacidadeinstalada'].to_numpy())
    yb.append(np.full(len(te), y_train.iloc[tr].mean()))       # baseline: média do treino do fold

y_pred = np.concatenate(yp)
y_true = np.concatenate(yt)
cap = np.concatenate(cp)

# ------------------------------------------------------------
# Métricas base
# ------------------------------------------------------------
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

erro_abs = np.abs(y_true - y_pred)
mask_cap = cap > 0
erro_pct_cap = np.where(mask_cap, erro_abs / np.where(mask_cap, cap, 1) * 100, np.nan)

nmae = np.nanmean(erro_pct_cap)
wape = erro_abs.sum() / y_true.sum() * 100

# Baseline ingênuo: prever sempre a média do treino
y_base = np.concatenate(yb)
rmse_base = np.sqrt(mean_squared_error(y_true, y_base))
mae_base = mean_absolute_error(y_true, y_base)

# ------------------------------------------------------------
# Métricas intuitivas
# ------------------------------------------------------------
# 1) Confiabilidade geral: quanto do volume gerado o modelo acerta
confiabilidade = max(0.0, 100 - wape)

# 2) Diferença percentual total: previsto vs real (soma de tudo)
total_real = y_true.sum()
total_prev = y_pred.sum()
dif_total_pct = (total_prev - total_real) / total_real * 100

# 3) Diferença percentual média por amostra (só onde o real > 0, evita divisão por zero)
mask_real = y_true > 0
dif_pct_amostra = np.abs(y_pred[mask_real] - y_true[mask_real]) / y_true[mask_real] * 100
dif_pct_mediana = np.median(dif_pct_amostra)

# 4) Taxa de acerto por faixa de erro (em % da capacidade instalada)
def taxa_acerto(tol_pct):
    return np.nanmean(erro_pct_cap <= tol_pct) * 100

# 5) Classificação textual da confiabilidade
def classificar(c):
    if c >= 90: return "EXCELENTE"
    if c >= 80: return "MUITO BOA"
    if c >= 70: return "BOA"
    if c >= 60: return "REGULAR"
    return "BAIXA"

# 6) Tendência: o modelo superestima ou subestima?
vies = "superestima" if dif_total_pct > 0 else "subestima"

# ------------------------------------------------------------
# Relatório
# ------------------------------------------------------------
print("=" * 60)
print(f"  RELATÓRIO DE DESEMPENHO DO MODELO — {len(y_true):,} amostras de teste")
print("=" * 60)

print(f"\n>> CONFIABILIDADE GERAL: {confiabilidade:.1f}%  [{classificar(confiabilidade)}]")
print(f"   O modelo acerta ~{confiabilidade:.0f}% do volume de energia gerada.")

print("\n>> PREVISTO vs REAL")
print(f"   Geração real total:     {total_real:>14,.0f} MW")
print(f"   Geração prevista total: {total_prev:>14,.0f} MW")
print(f"   Diferença total:        {dif_total_pct:>+13.2f}%  (o modelo {vies} levemente)")
print(f"   Erro típico por amostra: {dif_pct_mediana:.1f}% (mediana)")

print("\n>> ERRO EM MW")
print(f"   Em média, a previsão erra por {mae:.1f} MW (baseline ingênuo: {mae_base:.1f} MW)")
print(f"   Erro médio equivale a {nmae:.1f}% da capacidade instalada")

print("\n>> TAXA DE ACERTO (previsão dentro de X% da capacidade instalada)")
for tol in (0, 0.1, 1, 5, 10, 15, 20):
    t = taxa_acerto(tol)
    barra = "█" * int(t / 5) + "░" * (20 - int(t / 5))
    print(f"   até {tol:>3}%:  {barra}  {t:5.1f}%")

print("\n>> COMPARAÇÃO COM A MÉDIA")
print(f"   Melhora no erro (RMSE): {(1 - rmse / rmse_base) * 100:.1f}% melhor que o baseline")
print(f"   R² = {r2:.3f}  → o modelo explica {r2 * 100:.1f}% da variação da geração")

print("\n" + "=" * 60)

# ------------------------------------------------------------
# Salva as predições
# ------------------------------------------------------------
np.savetxt('conjunto_teste.predict', y_pred, fmt='%.3f')

### SVR:

In [ ]:
cat_cols = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),   # SVR não aceita NaN
        ('scaler', StandardScaler()),
    ]), num_cols),
])

pipeline = Pipeline([
    ('prep', preprocess),
    ('svr', SVR()),
])

model = TransformedTargetRegressor(
    regressor=pipeline,
    transformer=StandardScaler(),
)

param_grid = {
    'regressor__svr__kernel':  ['rbf'], # ['linear', 'rbf', 'poly', 'sigmoid', 'precomputed']
    'regressor__svr__C':       [1], # [1, 10, 100]
    'regressor__svr__epsilon': [0.1], # [0.01, 0.1, 0.5]
    'regressor__svr__gamma':   ['scale'], # ['scale', 0.01, 0.1]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv_splits,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

In [ ]:
grid_search.fit(X_train, y_train)
print(f"Melhores parâmetros: {grid_search.best_params_}")

In [ ]:
# ------------------------------------------------------------
# Resultado do CV
# ------------------------------------------------------------
print(f"Melhores parâmetros: {grid_search.best_params_}")
print(f"RMSE médio no CV (treino): {-grid_search.best_score_:.4f}")
print(f"Ano alvo: {ano_alvo}\n")

# ------------------------------------------------------------
# Predição fora da amostra: ref  # fração final de cada bloco que vira testeaz cada fold com os melhores parâmetros
# e junta os testes dos 5 folds (não há hold-out final)
# ------------------------------------------------------------
yp, yt, cp, yb = [], [], [], []
for tr, te in cv_splits:
    m = clone(grid_search.best_estimator_)
    m.fit(X_train.iloc[tr], y_train.iloc[tr])
    yp.append(np.clip(m.predict(X_train.iloc[te]), 0, None))   # geração não é negativa
    yt.append(y_train.iloc[te].to_numpy())
    cp.append(X_train.iloc[te]['val_capacidadeinstalada'].to_numpy())
    yb.append(np.full(len(te), y_train.iloc[tr].mean()))       # baseline: média do treino do fold

y_pred = np.concatenate(yp)
y_true = np.concatenate(yt)
cap = np.concatenate(cp)

# ------------------------------------------------------------
# Métricas base
# ------------------------------------------------------------
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

erro_abs = np.abs(y_true - y_pred)
mask_cap = cap > 0
erro_pct_cap = np.where(mask_cap, erro_abs / np.where(mask_cap, cap, 1) * 100, np.nan)

nmae = np.nanmean(erro_pct_cap)
wape = erro_abs.sum() / y_true.sum() * 100

# Baseline ingênuo: prever sempre a média do treino
y_base = np.concatenate(yb)
rmse_base = np.sqrt(mean_squared_error(y_true, y_base))
mae_base = mean_absolute_error(y_true, y_base)

# ------------------------------------------------------------
# Métricas intuitivas
# ------------------------------------------------------------
# 1) Confiabilidade geral: quanto do volume gerado o modelo acerta
confiabilidade = max(0.0, 100 - wape)

# 2) Diferença percentual total: previsto vs real (soma de tudo)
total_real = y_true.sum()
total_prev = y_pred.sum()
dif_total_pct = (total_prev - total_real) / total_real * 100

# 3) Diferença percentual média por amostra (só onde o real > 0, evita divisão por zero)
mask_real = y_true > 0
dif_pct_amostra = np.abs(y_pred[mask_real] - y_true[mask_real]) / y_true[mask_real] * 100
dif_pct_mediana = np.median(dif_pct_amostra)

# 4) Taxa de acerto por faixa de erro (em % da capacidade instalada)
def taxa_acerto(tol_pct):
    return np.nanmean(erro_pct_cap <= tol_pct) * 100

# 5) Classificação textual da confiabilidade
def classificar(c):
    if c >= 90: return "EXCELENTE"
    if c >= 80: return "MUITO BOA"
    if c >= 70: return "BOA"
    if c >= 60: return "REGULAR"
    return "BAIXA"

# 6) Tendência: o modelo superestima ou subestima?
vies = "superestima" if dif_total_pct > 0 else "subestima"

# ------------------------------------------------------------
# Relatório
# ------------------------------------------------------------
print("=" * 60)
print(f"  RELATÓRIO DE DESEMPENHO DO MODELO — {len(y_true):,} amostras de teste")
print("=" * 60)

print(f"\n>> CONFIABILIDADE GERAL: {confiabilidade:.1f}%  [{classificar(confiabilidade)}]")
print(f"   O modelo acerta ~{confiabilidade:.0f}% do volume de energia gerada.")

print("\n>> PREVISTO vs REAL")
print(f"   Geração real total:     {total_real:>14,.0f} MW")
print(f"   Geração prevista total: {total_prev:>14,.0f} MW")
print(f"   Diferença total:        {dif_total_pct:>+13.2f}%  (o modelo {vies} levemente)")
print(f"   Erro típico por amostra: {dif_pct_mediana:.1f}% (mediana)")

print("\n>> ERRO EM MW")
print(f"   Em média, a previsão erra por {mae:.1f} MW (baseline ingênuo: {mae_base:.1f} MW)")
print(f"   Erro médio equivale a {nmae:.1f}% da capacidade instalada")

print("\n>> TAXA DE ACERTO (previsão dentro de X% da capacidade instalada)")
for tol in (0, 0.1, 1, 5, 10, 15, 20):
    t = taxa_acerto(tol)
    barra = "█" * int(t / 5) + "░" * (20 - int(t / 5))
    print(f"   até {tol:>3}%:  {barra}  {t:5.1f}%")

print("\n>> COMPARAÇÃO COM A MÉDIA")
print(f"   Melhora no erro (RMSE): {(1 - rmse / rmse_base) * 100:.1f}% melhor que o baseline")
print(f"   R² = {r2:.3f}  → o modelo explica {r2 * 100:.1f}% da variação da geração")

print("\n" + "=" * 60)

# ------------------------------------------------------------
# Salva as predições
# ------------------------------------------------------------
np.savetxt('conjunto_teste.predict', y_pred, fmt='%.3f')

## Gradient Boosting:

In [ ]:
cat_cols = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
    ('num', 'passthrough', num_cols),
])

pipeline = Pipeline([
    ('prep', preprocess),   
    ('xgb', XGBRegressor(
        objective='reg:squarederror',
        tree_method='hist',
        random_state=42,
        n_jobs=-1,
    )),
])

param_grid = {
    'xgb__n_estimators':     [200], # [200, 500], 
    'xgb__learning_rate':    [0.03], # [0.03, 0.1],
    'xgb__max_depth':        [6], # [4, 6, 8],
    'xgb__subsample':        [0.7], # [0.7, 1.0],
    'xgb__colsample_bytree': [0.8], # [0.8, 1.0],
    'xgb__min_child_weight': [1], # [1, 5, 10],
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv_splits,
    scoring='neg_root_mean_squared_error',
    n_jobs=1,
    verbose=1,
    error_score='raise',
)

In [ ]:
grid_search.fit(X_train, y_train)
print(f"Melhores parâmetros: {grid_search.best_params_}")

In [ ]:
# ------------------------------------------------------------
# Resultado do CV
# ------------------------------------------------------------
print(f"Melhores parâmetros: {grid_search.best_params_}")
print(f"RMSE médio no CV (treino): {-grid_search.best_score_:.4f}")
print(f"Ano alvo: {ano_alvo}\n")

# ------------------------------------------------------------
# Predição fora da amostra: ref  # fração final de cada bloco que vira testeaz cada fold com os melhores parâmetros
# e junta os testes dos 5 folds (não há hold-out final)
# ------------------------------------------------------------
yp, yt, cp, yb = [], [], [], []
for tr, te in cv_splits:
    m = clone(grid_search.best_estimator_)
    m.fit(X_train.iloc[tr], y_train.iloc[tr])
    yp.append(np.clip(m.predict(X_train.iloc[te]), 0, None))   # geração não é negativa
    yt.append(y_train.iloc[te].to_numpy())
    cp.append(X_train.iloc[te]['val_capacidadeinstalada'].to_numpy())
    yb.append(np.full(len(te), y_train.iloc[tr].mean()))       # baseline: média do treino do fold

y_pred = np.concatenate(yp)
y_true = np.concatenate(yt)
cap = np.concatenate(cp)

# ------------------------------------------------------------
# Métricas base
# ------------------------------------------------------------
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

erro_abs = np.abs(y_true - y_pred)
mask_cap = cap > 0
erro_pct_cap = np.where(mask_cap, erro_abs / np.where(mask_cap, cap, 1) * 100, np.nan)

nmae = np.nanmean(erro_pct_cap)
wape = erro_abs.sum() / y_true.sum() * 100

# Baseline ingênuo: prever sempre a média do treino
y_base = np.concatenate(yb)
rmse_base = np.sqrt(mean_squared_error(y_true, y_base))
mae_base = mean_absolute_error(y_true, y_base)

# ------------------------------------------------------------
# Métricas intuitivas
# ------------------------------------------------------------
# 1) Confiabilidade geral: quanto do volume gerado o modelo acerta
confiabilidade = max(0.0, 100 - wape)

# 2) Diferença percentual total: previsto vs real (soma de tudo)
total_real = y_true.sum()
total_prev = y_pred.sum()
dif_total_pct = (total_prev - total_real) / total_real * 100

# 3) Diferença percentual média por amostra (só onde o real > 0, evita divisão por zero)
mask_real = y_true > 0
dif_pct_amostra = np.abs(y_pred[mask_real] - y_true[mask_real]) / y_true[mask_real] * 100
dif_pct_mediana = np.median(dif_pct_amostra)

# 4) Taxa de acerto por faixa de erro (em % da capacidade instalada)
def taxa_acerto(tol_pct):
    return np.nanmean(erro_pct_cap <= tol_pct) * 100

# 5) Classificação textual da confiabilidade
def classificar(c):
    if c >= 90: return "EXCELENTE"
    if c >= 80: return "MUITO BOA"
    if c >= 70: return "BOA"
    if c >= 60: return "REGULAR"
    return "BAIXA"

# 6) Tendência: o modelo superestima ou subestima?
vies = "superestima" if dif_total_pct > 0 else "subestima"

# ------------------------------------------------------------
# Relatório
# ------------------------------------------------------------
print("=" * 60)
print(f"  RELATÓRIO DE DESEMPENHO DO MODELO — {len(y_true):,} amostras de teste")
print("=" * 60)

print(f"\n>> CONFIABILIDADE GERAL: {confiabilidade:.1f}%  [{classificar(confiabilidade)}]")
print(f"   O modelo acerta ~{confiabilidade:.0f}% do volume de energia gerada.")

print("\n>> PREVISTO vs REAL")
print(f"   Geração real total:     {total_real:>14,.0f} MW")
print(f"   Geração prevista total: {total_prev:>14,.0f} MW")
print(f"   Diferença total:        {dif_total_pct:>+13.2f}%  (o modelo {vies} levemente)")
print(f"   Erro típico por amostra: {dif_pct_mediana:.1f}% (mediana)")

print("\n>> ERRO EM MW")
print(f"   Em média, a previsão erra por {mae:.1f} MW (baseline ingênuo: {mae_base:.1f} MW)")
print(f"   Erro médio equivale a {nmae:.1f}% da capacidade instalada")

print("\n>> TAXA DE ACERTO (previsão dentro de X% da capacidade instalada)")
for tol in (0, 0.1, 1, 5, 10, 15, 20):
    t = taxa_acerto(tol)
    barra = "█" * int(t / 5) + "░" * (20 - int(t / 5))
    print(f"   até {tol:>3}%:  {barra}  {t:5.1f}%")

print("\n>> COMPARAÇÃO COM A MÉDIA")
print(f"   Melhora no erro (RMSE): {(1 - rmse / rmse_base) * 100:.1f}% melhor que o baseline")
print(f"   R² = {r2:.3f}  → o modelo explica {r2 * 100:.1f}% da variação da geração")

print("\n" + "=" * 60)

# ------------------------------------------------------------
# Salva as predições
# ------------------------------------------------------------
np.savetxt('conjunto_teste.predict', y_pred, fmt='%.3f')

## TabPFN:

In [ ]:
# ------------------------------------------------------------
# Configuração (ajuste conforme o hardware)
# TabPFN não escala para o treino inteiro: cada fold usa uma subamostra
# como contexto e uma subamostra do teste para avaliar.
# ------------------------------------------------------------
USA_GPU      = torch.cuda.is_available()
N_CONTEXTO   = 10_000 if USA_GPU else 3_000   # linhas de treino por fold
N_TESTE_MAX  = 5_000  if USA_GPU else 1_500   # linhas de teste por fold
SEED         = 42
print(f"GPU disponível: {USA_GPU} | contexto={N_CONTEXTO:,} | teste/fold<={N_TESTE_MAX:,}")

cat_cols = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
    ('num', 'passthrough', num_cols),   # NaN é aceito nativamente pelo TabPFN
])

pipeline = Pipeline([
    ('prep', preprocess),
    ('tabpfn', TabPFNRegressor(
        # o ColumnTransformer coloca as categóricas nas primeiras colunas
        categorical_features_indices=list(range(len(cat_cols))),
        device='cuda' if USA_GPU else 'cpu',
        random_state=SEED,
    )),
])

param_grid = {
    'tabpfn__n_estimators': [4 if USA_GPU else 2],   # [2, 4, 8]
}

# ------------------------------------------------------------
# Folds subamostrados: mesmos folds temporais do RF (cv_splits),
# só que com N_CONTEXTO linhas no treino e N_TESTE_MAX no teste
# ------------------------------------------------------------
rng = np.random.default_rng(SEED)
cv_splits_pfn = [
    (np.sort(rng.choice(tr, size=min(N_CONTEXTO, len(tr)), replace=False)),
     np.sort(rng.choice(te, size=min(N_TESTE_MAX, len(te)), replace=False)))
    for tr, te in cv_splits
]

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv_splits_pfn,
    scoring='neg_root_mean_squared_error',
    n_jobs=1,        # um modelo por vez (GPU/RAM)
    refit=False,     # não refaz no treino completo: não cabe no TabPFN
    verbose=1
)


In [ ]:
grid_search.fit(X_train, y_train)
print(f"Melhores parâmetros: {grid_search.best_params_}")


In [ ]:
# ------------------------------------------------------------
# Resultado do CV
# ------------------------------------------------------------
print(f"Melhores parâmetros: {grid_search.best_params_}")
print(f"RMSE médio no CV (subamostra): {-grid_search.best_score_:.4f}")
print(f"Ano alvo: {ano_alvo}\n")

# ------------------------------------------------------------
# Predição fora da amostra: refaz cada fold (subamostrado) com os melhores
# parâmetros e junta os testes dos folds (não há hold-out final)
# ------------------------------------------------------------
yp, yt, cp, yb = [], [], [], []
for (tr, te), (tr_full, _) in zip(cv_splits_pfn, cv_splits):
    m = clone(pipeline).set_params(**grid_search.best_params_)   # refit=False: não há best_estimator_
    m.fit(X_train.iloc[tr], y_train.iloc[tr])
    yp.append(np.clip(m.predict(X_train.iloc[te]), 0, None))   # geração não é negativa
    yt.append(y_train.iloc[te].to_numpy())
    cp.append(X_train.iloc[te]['val_capacidadeinstalada'].to_numpy())
    yb.append(np.full(len(te), y_train.iloc[tr_full].mean()))  # baseline: média do treino completo do fold

y_pred = np.concatenate(yp)
y_true = np.concatenate(yt)
cap = np.concatenate(cp)

# ------------------------------------------------------------
# Métricas base
# ------------------------------------------------------------
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

erro_abs = np.abs(y_true - y_pred)
mask_cap = cap > 0
erro_pct_cap = np.where(mask_cap, erro_abs / np.where(mask_cap, cap, 1) * 100, np.nan)

nmae = np.nanmean(erro_pct_cap)
wape = erro_abs.sum() / y_true.sum() * 100

# Baseline ingênuo: prever sempre a média do treino
y_base = np.concatenate(yb)
rmse_base = np.sqrt(mean_squared_error(y_true, y_base))
mae_base = mean_absolute_error(y_true, y_base)

# ------------------------------------------------------------
# Métricas intuitivas
# ------------------------------------------------------------
# 1) Confiabilidade geral: quanto do volume gerado o modelo acerta
confiabilidade = max(0.0, 100 - wape)

# 2) Diferença percentual total: previsto vs real (soma de tudo)
total_real = y_true.sum()
total_prev = y_pred.sum()
dif_total_pct = (total_prev - total_real) / total_real * 100

# 3) Diferença percentual média por amostra (só onde o real > 0, evita divisão por zero)
mask_real = y_true > 0
dif_pct_amostra = np.abs(y_pred[mask_real] - y_true[mask_real]) / y_true[mask_real] * 100
dif_pct_mediana = np.median(dif_pct_amostra)

# 4) Taxa de acerto por faixa de erro (em % da capacidade instalada)
def taxa_acerto(tol_pct):
    return np.nanmean(erro_pct_cap <= tol_pct) * 100

# 5) Classificação textual da confiabilidade
def classificar(c):
    if c >= 90: return "EXCELENTE"
    if c >= 80: return "MUITO BOA"
    if c >= 70: return "BOA"
    if c >= 60: return "REGULAR"
    return "BAIXA"

# 6) Tendência: o modelo superestima ou subestima?
vies = "superestima" if dif_total_pct > 0 else "subestima"

# ------------------------------------------------------------
# Relatório
# ------------------------------------------------------------
print("=" * 60)
print(f"  RELATÓRIO DE DESEMPENHO DO TabPFN — {len(y_true):,} amostras de teste (subamostra)")
print("=" * 60)

print(f"\n>> CONFIABILIDADE GERAL: {confiabilidade:.1f}%  [{classificar(confiabilidade)}]")
print(f"   O modelo acerta ~{confiabilidade:.0f}% do volume de energia gerada.")

print("\n>> PREVISTO vs REAL")
print(f"   Geração real total:     {total_real:>14,.0f} MW")
print(f"   Geração prevista total: {total_prev:>14,.0f} MW")
print(f"   Diferença total:        {dif_total_pct:>+13.2f}%  (o modelo {vies} levemente)")
print(f"   Erro típico por amostra: {dif_pct_mediana:.1f}% (mediana)")

print("\n>> ERRO EM MW")
print(f"   Em média, a previsão erra por {mae:.1f} MW (baseline ingênuo: {mae_base:.1f} MW)")
print(f"   Erro médio equivale a {nmae:.1f}% da capacidade instalada")

print("\n>> TAXA DE ACERTO (previsão dentro de X% da capacidade instalada)")
for tol in (0, 0.1, 1, 5, 10, 15, 20):
    t = taxa_acerto(tol)
    barra = "█" * int(t / 5) + "░" * (20 - int(t / 5))
    print(f"   até {tol:>3}%:  {barra}  {t:5.1f}%")

print("\n>> COMPARAÇÃO COM A MÉDIA")
print(f"   Melhora no erro (RMSE): {(1 - rmse / rmse_base) * 100:.1f}% melhor que o baseline")
print(f"   R² = {r2:.3f}  → o modelo explica {r2 * 100:.1f}% da variação da geração")

print("\n" + "=" * 60)

# ------------------------------------------------------------
# Salva as predições
# ------------------------------------------------------------
np.savetxt('conjunto_teste_tabpfn.predict', y_pred, fmt='%.3f')